In [6]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.data import DataLoader

In [8]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

In [19]:
train_path=r"C:\Users\vpokh\Downloads\FIRE-SMOKE-DATASET\FIRE-SMOKE-DATASET\Train"
test_path=r"C:\Users\vpokh\Downloads\FIRE-SMOKE-DATASET\FIRE-SMOKE-DATASET\Test"
train_data = datasets.ImageFolder(root=train_path,transform=train_transform)
test_data = datasets.ImageFolder(root=test_path,transform=test_transform)
print("Classes:", train_data.classes)

Classes: ['Fire', 'Neutral', 'Smoke']


In [20]:
from torch.utils.data import random_split

train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size

train_dataset, val_dataset = random_split(train_data, [train_size, val_size])

In [26]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader   = DataLoader(test_data, batch_size=32, shuffle=False)

In [22]:
images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([32, 3, 224, 224]) torch.Size([32])


In [23]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(pretrained=True)

# Freeze early layers (optional but recommended)
for param in model.parameters():
    param.requires_grad = False

# Replace final layer for 3 classes
model.fc = nn.Linear(model.fc.in_features, 3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [24]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

In [25]:
epochs = 10

for epoch in range(epochs):

    # ---- TRAIN ----
    model.train()
    train_correct = 0
    train_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()

    train_acc = train_correct / len(train_dataset)

    # ---- VALIDATION ----
    model.eval()
    val_correct = 0
    val_loss = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()

    val_acc = val_correct / len(val_dataset)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")

Epoch 1
Train Loss: 48.0838, Train Acc: 0.7056
Val   Loss: 7.3976, Val   Acc: 0.8407
Epoch 2
Train Loss: 26.3741, Train Acc: 0.8708
Val   Loss: 5.8783, Val   Acc: 0.8722
Epoch 3
Train Loss: 20.3426, Train Acc: 0.9097
Val   Loss: 5.1314, Val   Acc: 0.8907
Epoch 4
Train Loss: 19.1305, Train Acc: 0.9028
Val   Loss: 4.9212, Val   Acc: 0.8926
Epoch 5
Train Loss: 16.2110, Train Acc: 0.9245
Val   Loss: 4.5286, Val   Acc: 0.8907
Epoch 6
Train Loss: 15.5543, Train Acc: 0.9204
Val   Loss: 4.8937, Val   Acc: 0.8907
Epoch 7
Train Loss: 15.8253, Train Acc: 0.9181
Val   Loss: 4.1390, Val   Acc: 0.9148
Epoch 8
Train Loss: 13.7203, Train Acc: 0.9333
Val   Loss: 4.0390, Val   Acc: 0.9093
Epoch 9
Train Loss: 13.5871, Train Acc: 0.9329
Val   Loss: 4.0626, Val   Acc: 0.9111
Epoch 10
Train Loss: 12.6711, Train Acc: 0.9356
Val   Loss: 4.1069, Val   Acc: 0.9167


In [27]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 89.08%


In [29]:
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": train_data.classes
}, "fire_model.pth")